In [ ]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster()
client = Client(cluster)

In [2]:
foldername = '/home/edavenport/analysis/vel-assim-manuscript/assimilation_results/spectra/'

In [3]:
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 13
import cmocean.cm as cmo
import xarray as xr
from open_tpose import tpose2012to2013
import numpy as np
import xarray as xr
from xmitgcm import open_mdsdataset

prefix = ['diag_state','diag_surf']
ds_tpose_noTAO = tpose2012to2013(prefix)

ds_tpose_noTAO['XC'] = ds_tpose_noTAO.XC.astype(float)
ds_tpose_noTAO['YC'] = ds_tpose_noTAO.YC.astype(float)
ds_tpose_noTAO['Z'] = ds_tpose_noTAO.Z.astype(float)
ds_tpose_noTAO['XG'] = ds_tpose_noTAO.XG.astype(float)
ds_tpose_noTAO['YC'] = ds_tpose_noTAO.YC.astype(float)

data_dir = '/data/SO3/edavenport/tpose6/sep2012/velocity_assim/run_iter22/'
grid_dir = '/data/SO6/TPOSE_diags/tpose6/grid_6/'

offset = 10
num_diags = 30+31+offset #sep, oct + 10 days
itPerFile = 72 # 1 day
intervals = range(itPerFile,itPerFile*num_diags,itPerFile)

ds = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-08-31 12:00:00',delta_t=1200)

num_diags = 30+31+offset# nov, dec (starting from nov 10)
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/nov2012/run_iter20/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-10-31 12:00:00',delta_t=1200)

ds_tpose_TAO = xr.concat([ds,ds_new],dim='time')

num_diags = 31+28+offset # jan, feb, (starting from jan 10) # add + offset to continue
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/jan2013/run_iter14/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-12-31 12:00:00',delta_t=1200)

ds_tpose_TAO = xr.concat([ds_tpose_TAO,ds_new],dim='time')

num_diags = 31+30+31+30+1 # mar, apr, may, june (starting from jan 10)
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/mar2013/run_iter16/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2013-03-01',delta_t=1200)

ds_tpose_TAO = xr.concat([ds_tpose_TAO,ds_new],dim='time')

ds_tpose_TAO['XC'] = ds_tpose_TAO.XC.astype(float)
ds_tpose_TAO['YC'] = ds_tpose_TAO.YC.astype(float)
ds_tpose_TAO['Z'] = ds_tpose_TAO.Z.astype(float)
ds_tpose_TAO['XG'] = ds_tpose_TAO.XG.astype(float)
ds_tpose_TAO['YC'] = ds_tpose_TAO.YC.astype(float)

mar2013/diags_daily/
may2013/diags_daily/
jul2013/diags_daily/
sep2013/diags_daily/
nov2013/diags_daily/
Days in 2012-2013: (should be 731)
731


In [4]:
ds_tpose_noTAO = ds_tpose_noTAO.sel(time=slice('2012-09-01','2013-06-30'))

In [41]:
def psd_for_plot_2d(data_array):
    # --- time spacing in seconds ---
    dt = (data_array.time[1] - data_array.time[0]).values / np.timedelta64(1, 's')
    fs = 1.0 / dt                     # sampling frequency (Hz)
    N = len(data_array)
    window = np.hanning(N)

    data_detrended = (data_array - data_array.mean(dim='time'))
    data_windowed = data_detrended * window[:,np.newaxis]
    fft_values = np.fft.rfft(data_windowed,axis=0)

    # --- PSD normalization ---
    U = (window**2).sum()       # window power
    psd_hz = (np.abs(fft_values) ** 2) / (U * fs)
    psd_hz[1:-1] *= 2 # variance preserving spectra
    psd_cpd = psd_hz / 86400

    freq_hz = np.fft.rfftfreq(N, d=dt)
    freq_dpc = 1/(freq_hz * 86400)        # cycles per day
    psd_dpc = psd_cpd / (freq_dpc[:,np.newaxis]**2)
    
    return fft_values, psd_hz, psd_dpc, freq_hz, freq_dpc

def psd_for_plot_1d(data_array):
    # --- time spacing in seconds ---
    dt = (data_array.time[1] - data_array.time[0]).values / np.timedelta64(1, 's')
    fs = 1.0 / dt                     # sampling frequency (Hz)
    N = len(data_array)
    window = np.hanning(N)

    data_detrended = (data_array - data_array.mean(dim='time'))
    data_windowed = data_detrended * window
    fft_values = np.fft.rfft(data_windowed,axis=0)

    # --- PSD normalization ---
    U = (window**2).sum()       # window power
    psd_hz = (np.abs(fft_values) ** 2) / (U * fs)
    psd_hz[1:-1] *= 2 # variance preserving spectra
    psd_cpd = psd_hz / 86400
    freq_hz = np.fft.rfftfreq(N, d=dt)
    freq_dpc = 1/(freq_hz * 86400)        # cycles per day
    psd_dpc = psd_cpd / (freq_dpc**2)
    
    return fft_values, psd_hz, psd_dpc, freq_hz, freq_dpc

In [32]:
import numba
import xgcm 
grid = xgcm.Grid(ds_tpose_TAO, periodic=False)
UVEL = grid.interp(ds_tpose_TAO.UVEL, 'X', boundary='fill')
grid = xgcm.Grid(ds_tpose_noTAO, periodic=False)
UVEL_noTAO = grid.interp(ds_tpose_noTAO.UVEL, 'X', boundary='fill')

In [27]:
grid = xgcm.Grid(ds_tpose_TAO, periodic=False)
VVEL = grid.interp(ds_tpose_TAO.VVEL, 'Y', boundary='fill')
grid = xgcm.Grid(ds_tpose_noTAO, periodic=False)
VVEL_noTAO = grid.interp(ds_tpose_noTAO.VVEL, 'Y', boundary='fill')

In [74]:
def integrated_power_spectrum(lon):
    _, _, psd, _, freq_dpc = psd_for_plot_2d(UVEL.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_2d(UVEL_noTAO.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = UVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values

    psd_u = xr.DataArray(
        psd,
        dims=['freq', 'Z'],
        coords={
            'freq': freq_dpc,
            'Z': z_levels
        }
    )

    psd_u_noTAO = xr.DataArray(
        psd_noTAO,
        dims=['freq', 'Z'],
        coords={
            'freq': freq_dpc,
            'Z': z_levels
        }
    )

    _, _, psd, _, freq_dpc = psd_for_plot_2d(VVEL.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_2d(VVEL_noTAO.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = VVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values

    psd_v = xr.DataArray(
        psd,
        dims=['freq', 'Z'],
        coords={
            'freq': freq_dpc,
            'Z': z_levels
        }
    )

    psd_v_noTAO = xr.DataArray(
        psd_noTAO,
        dims=['freq', 'Z'],
        coords={
            'freq': freq_dpc,
            'Z': z_levels
        }
    )

    P_int_u = (psd_u * ds_tpose_TAO.sel(Z=slice(0,-300)).drF).sum('Z')  # units: [X² m / Hz]
    P_int_u_noTAO = (psd_u_noTAO * ds_tpose_noTAO.sel(Z=slice(0,-300))['drF']).sum('Z')  # units: [X² m / Hz]
    P_int_v = (psd_v * ds_tpose_TAO.sel(Z=slice(0,-300))['drF']).sum('Z')  # units: [X² m / Hz]
    P_int_v_noTAO = (psd_v_noTAO * ds_tpose_noTAO.sel(Z=slice(0,-300))['drF']).sum('Z')  # units: [X² m / Hz]

    return P_int_u, P_int_u_noTAO, P_int_v, P_int_v_noTAO, freq_dpc

In [ ]:
from matplotlib.colors import LogNorm
from matplotlib import ticker
lons = [190.0, 205.0, 220.0, 235.0, 250.0]
cbar_kwargs={
        'ticks': ticker.LogLocator(base=10, subs=[1.0]),  # ticks at 10^n
        'format': ticker.LogFormatterMathtext(),           # renders as 10^n
    }
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(12, 18), sharex=True)
# log_levels = np.logspace(-5, -2, 50)
# norm = LogNorm(vmin=vmin, vmax=vmax)
# levels = np.linspace(1e-5,1e-3,100)
levels = np.logspace(-4.5, -2.5, 100)

for lon, ax in zip(lons,axes.flatten()[::2]):
    _, _, psd, _, freq_dpc = psd_for_plot_2d(VVEL.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = VVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_v = xr.DataArray(psd[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_v.plot.contourf(levels=levels,x='freq', y='Z', ax=ax, cmap='inferno', norm=LogNorm(vmin=levels.min(), vmax=levels.max()), cbar_kwargs=cbar_kwargs)
    ax.set_xlim(10,60)
    ax.set_xlabel('')
    ax.set_ylabel('Depth (m)')
    ax.set_title('{}°W TPOSE-Vel'.format(abs(int(lon-360))))

cbar_kwargs={
        'ticks': ticker.LogLocator(base=10, subs=[1.0]),  # ticks at 10^n
        'format': ticker.LogFormatterMathtext(),           # renders as 10^n
        'label': '(m$^2$ s$^{-2}$ / cpd)'
    }
for lon, ax in zip(lons,axes.flatten()[1::2]):
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_2d(VVEL_noTAO.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = VVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_v_noTAO = xr.DataArray(psd_noTAO[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_v_noTAO.plot.contourf(levels=levels,x='freq', y='Z', ax=ax, cmap='inferno',norm=LogNorm(vmin=levels.min(), vmax=levels.max()), cbar_kwargs=cbar_kwargs)
    ax.set_xlim(10,60)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title('{}°W TPOSE-noVel'.format(abs(int(lon-360))))

axes[4,0].set_xlabel('Frequency (cycles per day)')
axes[4,1].set_xlabel('Frequency (cycles per day)')

plt.tight_layout()
fig.savefig(foldername + 'psd_v_5lon.png',dpi=300)

In [ ]:

from matplotlib.colors import LogNorm
lons = [190.0, 220.0, 250.0]

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16, 10), sharex=True)
# log_levels = np.logspace(-5, -2, 50)
# norm = LogNorm(vmin=vmin, vmax=vmax)
# levels = np.linspace(1e-5,1e-3,100)
levels = np.logspace(-4.5, -2, 100)
cbar_kwargs={
        'ticks': ticker.LogLocator(base=10, subs=[1.0]),  # ticks at 10^n
        'format': ticker.LogFormatterMathtext(),           # renders as 10^n
        'label': '(m$^2$ s$^{-2}$ / cpd)'
    }
db_levels = np.linspace(-10,10,100)

for lon, ax in zip(lons,axes.flatten()[::3]):
    _, _, psd, _, freq_dpc = psd_for_plot_2d(VVEL.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = VVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_v = xr.DataArray(psd[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_v.plot.contourf(levels=levels,x='freq', y='Z', ax=ax, cmap='inferno', norm=LogNorm(vmin=levels.min(), vmax=levels.max()),cbar_kwargs=cbar_kwargs)
    ax.set_xlim(10,60)
    ax.set_xlabel('')
    ax.set_ylabel('Depth (m)')
    ax.set_title('{}°W TPOSE-Vel'.format(abs(int(lon-360))))

for lon, ax in zip(lons,axes.flatten()[1::3]):
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_2d(VVEL_noTAO.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = VVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_v_noTAO = xr.DataArray(psd_noTAO[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_v_noTAO.plot.contourf(levels=levels,x='freq', y='Z', ax=ax, cmap='inferno',norm=LogNorm(vmin=levels.min(), vmax=levels.max()),cbar_kwargs=cbar_kwargs)
    ax.set_xlim(10,60)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title('{}°W TPOSE-noVel'.format(abs(int(lon-360))))

for lon, ax in zip(lons,axes.flatten()[2::3]):
    _, _, psd_b, _, freq_dpc = psd_for_plot_2d(VVEL.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    _, _, psd_a, _, freq_dpc = psd_for_plot_2d(VVEL_noTAO.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    psd_ratio = 10 * np.log10(psd_b / psd_a)
    z_levels = VVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_ratio = xr.DataArray(psd_ratio[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_ratio.plot.contourf(levels=db_levels,x='freq', y='Z', ax=ax, cmap='RdBu_r', vmin=-10, vmax=10, cbar_kwargs={'label': 'dB','ticks': np.arange(-10, 11, 5)})
    ax.set_xlim(10,60)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title('{}°W'.format(abs(int(lon-360))))

axes[2,0].set_xlabel('Frequency (cycles per day)')
axes[2,1].set_xlabel('Frequency (cycles per day)')
axes[2,2].set_xlabel('Frequency (cycles per day)')

plt.tight_layout()
fig.savefig(foldername + 'psd_v_3lon_dB.png',dpi=300)

In [ ]:

from matplotlib.colors import LogNorm
lons = [190.0, 220.0, 250.0]

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16, 10), sharex=True)
# log_levels = np.logspace(-5, -2, 50)
# norm = LogNorm(vmin=vmin, vmax=vmax)
# levels = np.linspace(1e-5,1e-3,100)
levels = np.logspace(-4.5, -2, 100)
cbar_kwargs={
        'ticks': ticker.LogLocator(base=10, subs=[1.0]),  # ticks at 10^n
        'format': ticker.LogFormatterMathtext(),           # renders as 10^n
        'label': '(m$^2$ s$^{-2}$ / cpd)'
    }
db_levels = np.linspace(-10,10,100)

for lon, ax in zip(lons,axes.flatten()[::3]):
    _, _, psd, _, freq_dpc = psd_for_plot_2d(UVEL.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = UVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_v = xr.DataArray(psd[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_v.plot.contourf(levels=levels,x='freq', y='Z', ax=ax, cmap='inferno', norm=LogNorm(vmin=levels.min(), vmax=levels.max()),cbar_kwargs=cbar_kwargs)
    ax.set_xlim(10,120)
    ax.set_xlabel('')
    ax.set_ylabel('Depth (m)')
    ax.set_title('{}°W TPOSE-Vel'.format(abs(int(lon-360))))

for lon, ax in zip(lons,axes.flatten()[1::3]):
    _, _, psd_noTAO, _, freq_dpc = psd_for_plot_2d(UVEL_noTAO.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    z_levels = UVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_v_noTAO = xr.DataArray(psd_noTAO[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_v_noTAO.plot.contourf(levels=levels,x='freq', y='Z', ax=ax, cmap='inferno',norm=LogNorm(vmin=levels.min(), vmax=levels.max()),cbar_kwargs=cbar_kwargs)
    ax.set_xlim(10,120)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title('{}°W TPOSE-noVel'.format(abs(int(lon-360))))

for lon, ax in zip(lons,axes.flatten()[2::3]):
    _, _, psd_b, _, freq_dpc = psd_for_plot_2d(UVEL.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    _, _, psd_a, _, freq_dpc = psd_for_plot_2d(UVEL_noTAO.sel(YC=0,XC=lon,method='nearest').sel(Z=slice(0,-300)))
    psd_ratio = 10 * np.log10(psd_b / psd_a)
    z_levels = UVEL.sel(YC=0, XC=lon, method='nearest').sel(Z=slice(0,-300)).Z.values
    psd_ratio = xr.DataArray(psd_ratio[1:],dims=['freq', 'Z'],coords={'freq': freq_dpc[1:],'Z': z_levels})
    psd_ratio.plot.contourf(levels=db_levels,x='freq', y='Z', ax=ax, cmap='RdBu_r', vmin=-10, vmax=10, cbar_kwargs={'label': 'dB','ticks': np.arange(-10, 11, 5)})
    ax.set_xlim(10,120)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title('{}°W'.format(abs(int(lon-360))))

axes[2,0].set_xlabel('Frequency (cycles per day)')
axes[2,1].set_xlabel('Frequency (cycles per day)')
axes[2,2].set_xlabel('Frequency (cycles per day)')

plt.tight_layout()
fig.savefig(foldername + 'psd_u_3lon_dB.png',dpi=300)

In [ ]:
client.shutdown()
cluster.close()
client.close()